*We are merging two datasets— the main dataset is longitudinal data on medical expenses, demographic info, etc. for each person. The second dataset has information on the health conditions of each person*

In [2]:
import numpy as np
import pandas as pd

# Datasets
# ================================================

# MEPS HC-257: MEPS Panel 28 Longitudinal Data File: https://meps.ahrq.gov/mepsweb/data_stats/download_data_files_detail.jsp?cboPufNumber=HC-257&prfricon=yes&utm_source=chatgpt.com
    # each row is a unique person
    # key = dupersid — a primary unique person-level identifier
main_df = pd.read_stata('main.dta')

# 2023 Medical Conditions File: https://meps.ahrq.gov/mepsweb/data_stats/download_data_files_results.jsp?cboDataYear=All&cboDataTypeY=1%2CHousehold+Full+Year+File&buttonYearandDataType=Search&cboPufNumber=All&SearchTitle=Medical+Conditions
    # each row is a unique personXcondition combo - each row is NOT a unique person
    # key = dupersid — a primary unique person-level identifier
conditions_df = pd.read_stata('conditions.dta')

In [3]:
# Data Cleaning Helper Fxs
# ================================================
def code(s):
    """Leading numeric code from a MEPS value."""
    return pd.to_numeric(
        s.astype('string').str.extract(r'^\s*(-?\d+(?:\.\d+)?)')[0], errors='coerce'
    )

def num(s):
    """Numeric variable; MEPS negative missing codes -> NaN."""
    x = code(s)
    return x.where(x >= 0)

def yesno(s):
    """MEPS 1 = Yes, 2 = No -> 1/0; missing codes -> NaN."""
    return code(s).map({1: 1.0, 2: 0.0})

def label(s):
    """Categorical variable as its text label (code stripped); missing codes -> NaN."""
    txt = s.astype('string').str.strip()
    lab = txt.str.replace(r'^-?\d+(?:\.\d+)?\s*', '', regex=True)
    lab = lab.where(lab.str.len() > 0, txt)
    return lab.where(code(s) >= 0).astype('category')

In [4]:
# Variables Selected from main_df
    # Predictors are 2023 datapoints; outcome we wanna predict is 2024 healthcare costs— the tot_exp_y2 column
    # You can cross check the capitalized- names (the og column names) with the definitions in their data dict: https://meps.ahrq.gov/mepsweb/data_stats/download_data_files_codebook.jsp?PUFId=H257&utm_source=chatgpt.com
# ================================================

VARIABLES = {
    'outcome': {
        'tot_exp_y2': ('TOTEXPY2', num), # Total medical expidenture in 2024-- the outcome variable!
    },
    'A_demographics': { # Demographic info about each person
        'age':                ('AGEY1X',   num),
        'sex':                ('SEX',      label),
        'race_eth':           ('RACETHX',  label),
        'marital':            ('MARRYY1X', label),
        'education':          ('HIDEG',    label),
        'region':             ('REGIONY1', label),
        'employment':         ('EMPST2H',  label),
        'poverty_cat':        ('POVCATY1', label),
        'poverty_pct':        ('POVLEVY1', num),
        'insurance':          ('INSCOVY1', label),
        'medicaid':           ('MCDEVY1',  yesno),
        'medicare':           ('MCREVY1',  yesno),
        'uninsured_all_year': ('UNINSY1',  yesno),
    },
    'B_prior_use': { # Medical related expenses by each person
        'tot_exp_y1':       ('TOTEXPY1', num), # Total medical expidenture in 2023-- twenty twenty THREE
        'ob_exp_y1':        ('OBVEXPY1', num),
        'er_exp_y1':        ('ERTEXPY1', num),
        'ip_exp_y1':        ('IPTEXPY1', num),
        'rx_exp_y1':        ('RXEXPY1',  num),
        'hh_exp_y1':        ('HHAEXPY1', num),
        'oop_exp_y1':       ('TOTSLFY1', num),   # paid out of pocket
        'ob_visits_y1':     ('OBTOTVY1', num),
        'op_visits_y1':     ('OPTOTVY1', num),
        'er_visits_y1':     ('ERTOTY1',  num),
        'ip_discharges_y1': ('IPDISY1',  num),
        'ip_nights_y1':     ('IPNGTDY1', num),
        'rx_fills_y1':      ('RXTOTY1',  num),
        'hh_days_y1':       ('HHTOTDY1', num),
    },
    'C_medical': { # Person is asked if they were diagnosed with each of the illnesses
        'dx_hibp':           ('HIBPDXY1',     yesno),
        'dx_chd':            ('CHDDXY1',      yesno),
        'dx_angina':         ('ANGIDXY1',     yesno),
        'dx_mi':             ('MIDXY1',       yesno),
        'dx_oth_heart':      ('OHRTDXY1',     yesno),
        'dx_stroke':         ('STRKDXY1',     yesno),
        'dx_emphysema':      ('EMPHDXY1',     yesno),
        'dx_bronchitis':     ('CHBRON1',      yesno),
        'dx_chol':           ('CHOLDXY1',     yesno),
        'dx_cancer':         ('CANCERY1',     yesno),
        'dx_diabetes':       ('DIABDXY1_M18', yesno),
        'dx_arthritis':      ('ARTHDXY1',     yesno),
        'dx_asthma':         ('ASTHDXY1',     yesno),
        'self_rated_health': ('RTHLTH2',      num),    # 1 = Excellent ... 5 = Poor
        'self_rated_mental': ('MNHLTH2',      num),    # 1 = Excellent ... 5 = Poor
        'any_limitation':    ('ANYLMIY1',     yesno),
        'adl_help':          ('ADLHLP1',      yesno),
        'iadl_help':         ('IADLHP1',      yesno),
        'walk_limit':        ('WLKLIM1',      yesno),
        'cog_limit':         ('COGLIM1',      yesno),
        'social_limit':      ('SOCLIM1',      yesno),
        'physical_score':    ('VPCS2',        num),    # SF-12 physical; higher = better
        'mental_score':      ('VMCS2',        num),    # SF-12 mental; higher = better
        'k6_distress':       ('K6SUM2',       num),    # psychological distress; higher = worse
        'phq2_depress':      ('PHQ22',        num),    # depression screen; higher = worse
    },
    'D_engagement': { # Person is asked if they partook in these preventative care actions, like visiting the doctor routinely etc.
        'has_usual_source':  ('HAVEUS2',   yesno),
        'routine_checkup':   ('ADRTCR2',   yesno),
        'office_visits_saq': ('ADAPPT2',   num),     # 0 = none ... 6 = 10 or more
        'needed_specialist': ('ADSPCL2',   yesno),
        'care_rating':       ('ADHECR2',   num),     # 0–10; missing if no care received
        'delayed_care_cost': ('DLAYCA2',   yesno),
        'unafford_care':     ('AFRDCA2',   yesno),
        'delayed_rx_cost':   ('DLAYPM2',   yesno),
        'unafford_rx':       ('AFRDPM2',   yesno),
        'covid_vax_ever':    ('COVAXEVR1', yesno),
    },
    'E_behavior_attitudes': { # # Person is asked some behavioral health questions
        'smoker':            ('ADSMOK2', yesno),
        'no_need_insurance': ('ADINSA2', label),   # agree/disagree scale
        'risk_taker':        ('ADOVER2', label),   # agree/disagree scale
    },
}






# Now build the person-by- person level dataframe
# =============================================================
raw = main_df.copy()
raw['DUPERSID'] = raw['DUPERSID'].astype(str).str.strip()

df = pd.DataFrame({'dupersid': raw['DUPERSID']})
for block, variables in VARIABLES.items():
    for new_name, (meps_name, clean_fn) in variables.items():
        df[new_name] = clean_fn(raw[meps_name])

# Derived variables
df['log_tot_exp_y2'] = np.log1p(df['tot_exp_y2'])
dx_cols = [c for c in VARIABLES['C_medical'] if c.startswith('dx_')]
df['n_chronic'] = df[dx_cols].sum(axis=1, min_count=1)   # count of "yes" diagnoses

/opt/miniconda3/envs/ds6021/lib/python3.12/site-packages/pandas/core/arrays/masked.py:788: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs2, **kwargs)


In [5]:
# Additional information selected from conditions_df
# =============================================================
BODY_SYSTEMS = {
    'BLD': 'blood', 'CIR': 'circulatory', 'DEN': 'dental', 'DIG': 'digestive',
    'EAR': 'ear', 'END': 'endocrine', 'EXT': 'external_cause', 'EYE': 'eye',
    'FAC': 'health_factors', 'GEN': 'genitourinary', 'INF': 'infectious',
    'INJ': 'injury', 'MAL': 'congenital', 'MBD': 'mental_behavioral',
    'MUS': 'musculoskeletal', 'NEO': 'neoplasm', 'NVS': 'nervous',
    'PNL': 'perinatal', 'PRG': 'pregnancy', 'RSP': 'respiratory',
    'SKN': 'skin', 'SYM': 'symptoms',
}

cond = conditions_df.copy()
cond['DUPERSID'] = cond['DUPERSID'].astype(str).str.strip()

# Long format: one row per (person, condition, CCSR category); a condition can have up to 4
ccsr_long = cond.melt(id_vars=['DUPERSID', 'CONDIDX'],
                      value_vars=['CCSR1X', 'CCSR2X', 'CCSR3X', 'CCSR4X'],
                      value_name='ccsr')
ccsr_long['ccsr'] = ccsr_long['ccsr'].astype(str).str.strip().str.upper()
ccsr_long = (ccsr_long[ccsr_long['ccsr'].str.match(r'^[A-Z]{3}\d{3}$')]   # drop blanks/missing codes
             .drop_duplicates(['DUPERSID', 'CONDIDX', 'ccsr']))
ccsr_long['system'] = ccsr_long['ccsr'].str[:3].map(BODY_SYSTEMS)

# (a) Total number of reported conditions
n_conditions = cond.groupby('DUPERSID')['CONDIDX'].nunique().rename('n_conditions')

# (b) Number of conditions in each body system
by_system = ccsr_long.dropna(subset=['system']).drop_duplicates(['DUPERSID', 'CONDIDX', 'system'])
by_system = pd.crosstab(by_system['DUPERSID'], by_system['system']).add_prefix('n_cond_')

# (c) 0/1 flags for individual CCSR categories that are common in our panel
panel_ids = set(df['dupersid'])
flags = pd.crosstab(ccsr_long['DUPERSID'], ccsr_long['ccsr']).clip(upper=1)
flags = flags[flags.index.isin(panel_ids)]
flags = flags.loc[:, flags.sum() / len(panel_ids) >= 0.02].add_prefix('ccsr_') # 18+ and keep conditions that atleast 2% of people have

cond_features = pd.concat([n_conditions, by_system, flags], axis=1)
cond_cols = cond_features.columns.tolist()

# Merge onto the main file; people with no condition records get 0s
df = df.merge(cond_features, left_on='dupersid', right_index=True, how='left')
df[cond_cols] = df[cond_cols].fillna(0).astype(int)

In [6]:

# =============================================================
# 6. Analytic sample: adults with a valid 2024 spending value
# =============================================================
analytic_df = df[(df['age'] >= 18) & df['tot_exp_y2'].notna()].reset_index(drop=True)
analytic_df['high_cost_y2'] = (
    analytic_df['tot_exp_y2'] >= analytic_df['tot_exp_y2'].quantile(0.8)
).astype(int)                                             # top 20% of 2024 spenders

# =============================================================
# 7. Variable blocks (for summary tables and modeling)
# =============================================================
OUTCOME = 'log_tot_exp_y2'
BLOCKS = {block: list(v) for block, v in VARIABLES.items() if block != 'outcome'}
BLOCKS['C_medical'].append('n_chronic')
BLOCKS['F_conditions'] = cond_cols

# =============================================================
# 8. Sanity checks
# =============================================================
print(f"Analytic sample: {len(analytic_df):,} adults, {analytic_df.shape[1]} columns")
for block, cols in BLOCKS.items():
    print(f"  {block}: {len(cols)} variables")
print(f"\nCCSR flags kept (>= {0.02:.0%} prevalence): {flags.shape[1]}")
print("\nMost-missing variables (%):")
print((analytic_df.isna().mean() * 100).round(1).sort_values(ascending=False).head(15))

Analytic sample: 7,079 adults, 141 columns
  A_demographics: 13 variables
  B_prior_use: 14 variables
  C_medical: 26 variables
  D_engagement: 10 variables
  E_behavior_attitudes: 3 variables
  F_conditions: 71 variables

CCSR flags kept (>= 2% prevalence): 48

Most-missing variables (%):
care_rating          49.1
needed_specialist    33.6
k6_distress          33.1
phq2_depress         32.8
smoker               32.8
office_visits_saq    32.7
routine_checkup      32.6
risk_taker           32.5
no_need_insurance    32.3
mental_score         31.0
physical_score       30.7
has_usual_source      3.2
dx_bronchitis         1.6
covid_vax_ever        1.1
cog_limit             0.9
dtype: float64


/var/folders/6k/v3jzfsqx1jl_kk70v04s4kww0000gn/T/ipykernel_42054/1460507109.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  analytic_df['high_cost_y2'] = (


In [8]:
analytic_df.to_csv('meps_health_expenditures_v1.csv')